# Hybrid BM25 + MiniLM + FAISS

Iterations 18 and 19: first compare original MiniLM under the same abstention cutoffs as enrichment, then combine the stronger lexical retriever with original MiniLM. This notebook reads saved train results; no reranker, LLM, or test evaluation is involved.

In [2]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
rows=[json.loads(line) for line in (ROOT/'experiments/semantic_log.jsonl').read_text(encoding='utf-8').splitlines()]
control=next(row for row in reversed(rows) if row['iteration']==18)
run=next(row for row in reversed(rows) if row['iteration']==19)
run['config']

{'name': 'hybrid_bm25_minilm_rrf',
 'retriever_type': 'hybrid',
 'bm25_config': 'configs/bm25_focused_stemmed.json',
 'dense_config': 'semantic_configs/minilm.json',
 'candidate_pool': 20,
 'rrf_constant': 60,
 'bm25_weight': 0.75}

## Rank fusion

Retrieve 20 candidates per component. For each original source span, sum `weight / (60 + rank)`: BM25 weight 0.75 and dense weight 0.25. Exact duplicate source spans merge. These weights were declared before scoring because BM25 was stronger. RRF scores are not probabilities. Reference: https://learn.microsoft.com/en-us/azure/search/hybrid-search-ranking .

In [3]:
table=[]
for name,result in run['metrics'].items():
    m=next(m for m in result['retrieval'] if m['k']==5)
    table.append(dict(version=name,precision=m['precision'],recall=m['recall'],benchmark_precision=m['benchmark_precision'],benchmark_recall=m['benchmark_recall'],hit_rate=m['hit'],correct_abstention=result['correct_abstention'],false_abstention=result['false_abstention']))
pd.DataFrame(table).set_index('version').style.format('{:.2%}')

,precision,recall,benchmark_precision,benchmark_recall,hit_rate,correct_abstention,false_abstention
version,,,,,,,
bm25,7.88%,67.01%,8.38%,76.60%,74.29%,0.00%,0.00%
dense,4.03%,35.75%,4.19%,38.90%,46.43%,0.00%,0.00%
candidate,7.14%,66.23%,7.52%,75.16%,75.71%,0.00%,0.00%


## Dense control and hybrid cutoff trials

Both use the same threshold grid. For hybrid, the gate uses the top dense cosine score, so it can reject queries with useful lexical support. The 5% false-abstention limit applies alongside recall and precision protections. These are train tuning results.

In [4]:
table=[]
for name,record in [('dense_control',control),('hybrid',run)]:
    for t in record['threshold_trials']:
        m=next(m for m in t['metrics']['retrieval'] if m['k']==5)
        table.append(dict(version=name,threshold=t['threshold'],precision=m['precision'],recall=m['recall'],correct_abstention=t['metrics']['correct_abstention'],false_abstention=t['metrics']['false_abstention'],accepted=t['accepted']))
pd.DataFrame(table)

,version,threshold,precision,recall,correct_abstention,false_abstention,accepted
0,dense_control,0.40,0.040341,0.357489,0.000000,0.000000,False
1,dense_control,0.45,0.040341,0.357489,0.000000,0.000000,False
2,dense_control,0.50,0.040341,0.357489,0.130435,0.000000,True
3,dense_control,0.55,0.040341,0.357489,0.391304,0.007143,True
4,dense_control,0.60,0.038154,0.339046,0.695652,0.078571,False
5,dense_control,0.65,0.026758,0.236831,0.869565,0.371429,False
6,hybrid,0.40,0.071414,0.662276,0.000000,0.000000,False
7,hybrid,0.45,0.071414,0.662276,0.000000,0.000000,False
8,hybrid,0.50,0.071414,0.662276,0.130435,0.000000,True
9,hybrid,0.55,0.071166,0.655133,0.391304,0.007143,True


In [5]:
print('Dense control cutoff:',control['selected_threshold'])
print('Hybrid cutoff:',run['selected_threshold'])
print('Hybrid accepted against BM25:',run['accepted'],run['checks'])
print('Test evaluations:',run['test_evaluation_runs'])
print('Failure categories:',run['diagnosis_after'])

Dense control cutoff: 0.55
Hybrid cutoff: 0.55
Hybrid accepted against BM25: False {'recall_gain': False, 'precision': False, 'false_abstention': True, 'correct_abstention': True}
Test evaluations: 0
Failure categories: {'complete_coverage': 76, 'right_document_wrong_section': 34, 'partial_coverage': 30}


## Reproduce and modify

Run `python manage.py evaluate_hybrid` to repeat train evaluation with cached dense vectors. Settings are in `semantic_configs/hybrid_rrf.json`. Full predictions and every threshold attempt are in `results/iteration_19_hybrid/`. Modify a configuration only as a separately logged experiment. The next stage adds a reranker to this candidate pool.